# Bronze Layer — Raw Ingestion
**Storage:** Unity Catalog managed table `DE_1.bronze_healthcare`
- Data → Parquet files inside `ingestion_date=YYYY-MM-DD/` partition folders
- ACID log → `_delta_log/` JSON transaction files
- Running twice will NOT duplicate data — checks `source_file` column to prevent re-ingestion


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
import re

# Unity Catalog schema for serverless compute
spark.sql("CREATE SCHEMA IF NOT EXISTS de_1")
spark.sql("USE de_1")


DataFrame[]

## Widget
Business reason: ADF or a job trigger can pass a different source file without editing the notebook.


In [0]:
dbutils.widgets.text("source_path", "/Volumes/workspace/default/practice/healthcare_2019.csv", "Source File Path")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = auto)")

SOURCE_PATH = dbutils.widgets.get("source_path")
BATCH_ID    = dbutils.widgets.get("batch_id") or str(spark.sql("SELECT unix_timestamp()").collect()[0][0])

print(f"Source  : {SOURCE_PATH}")
print(f"Batch ID: {BATCH_ID}")


Source  : /Volumes/workspace/default/practice/healthcare_2019.csv
Batch ID: 1782137406


## 1 · File Format Validation & Error Handling
Unsupported formats are logged to quarantine Delta table — pipeline never fails silently.


In [0]:
SUPPORTED_FORMATS = {"csv", "json", "orc", "xlsx"}

def get_ext(path):
    return path.rsplit(".", 1)[-1].lower()

def read_raw(path):
    ext = get_ext(path)

    if ext not in SUPPORTED_FORMATS:
        spark.createDataFrame([{
            "source_file":  path,
            "error_reason": f"Unsupported format: .{ext}",
            "logged_at":    str(spark.sql("SELECT current_timestamp()").collect()[0][0])
        }]).write.format("delta").mode("append") \
          .option("mergeSchema", "true") \
          .saveAsTable("de_1.quarantine")
        raise ValueError(f"Unsupported format '.{ext}' — supported: {SUPPORTED_FORMATS}")

    if ext == "csv":
        df = (spark.read
                .option("header", "true")
                .option("inferSchema", "false")
                .option("mode", "PERMISSIVE")
                .option("columnNameOfCorruptRecord", "_corrupt_record")
                .csv(path))
    elif ext == "json":
        df = (spark.read
                .option("mode", "PERMISSIVE")
                .option("columnNameOfCorruptRecord", "_corrupt_record")
                .json(path))
    elif ext == "orc":
        df = spark.read.orc(path)
    elif ext == "xlsx":
        df = (spark.read
                .format("com.crealytics.spark.excel")
                .option("header", "true")
                .option("inferSchema", "false")
                .load(path))
    else:
        df = None

    if df is not None and df.count() == 0:
        spark.createDataFrame([{
            "source_file":  path,
            "error_reason": "Empty file",
            "logged_at":    str(spark.sql("SELECT current_timestamp()").collect()[0][0])
        }]).write.format("delta").mode("append") \
          .option("mergeSchema", "true") \
          .saveAsTable("de_1.quarantine")
        raise ValueError("Inserted file is empty — no rows found.")

    return df

## 2 · File Status Check
Checks if the source file already exists in bronze_healthcare. If found → upsert (update existing). If not found → insert new.


In [0]:
# Check if source file already exists in bronze_healthcare
try:
    existing = spark.sql(f"""
        SELECT COUNT(*) as cnt 
        FROM de_1.bronze_healthcare 
        WHERE source_file = '{SOURCE_PATH}'
    """).collect()[0]["cnt"]
    
    if existing > 0:
        print(f"UPDATE MODE — '{SOURCE_PATH}' already exists ({existing} rows found). Will upsert.")
    else:
        print(f"INSERT MODE — '{SOURCE_PATH}' not found. Will insert new records.")
except Exception as e:
    # Table doesn't exist yet — first load
    print(f"FIRST LOAD — Bronze table not found. Creating new table.")


UPDATE MODE — '/Volumes/workspace/default/practice/healthcare_2019.csv' already exists (7589 rows found). Will upsert.


## 3 · Read Raw File


In [0]:
raw_df = read_raw(SOURCE_PATH)

# Quarantine corrupt rows (PERMISSIVE mode captures them in _corrupt_record)
if "_corrupt_record" in raw_df.columns:
    corrupt_df = raw_df.filter(F.col("_corrupt_record").isNotNull()) \
                       .withColumn("source_file",   F.lit(SOURCE_PATH)) \
                       .withColumn("error_reason",  F.lit("parse_error")) \
                       .withColumn("logged_at",     F.current_timestamp())
    c = corrupt_df.count()
    if c > 0:
        corrupt_df.write.format("delta").mode("append") \
                  .option("mergeSchema", "true").saveAsTable("de_1.quarantine")
        print(f"Quarantined {c} corrupt rows")
    raw_df = raw_df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

print(f"Raw rows: {raw_df.count()}")


Raw rows: 7589


## 4 · Schema Drift Detection
New columns in source → kept via mergeSchema (schema evolution).
Missing columns → filled with null so downstream schema stays consistent.


In [0]:
EXPECTED_COLS = {
    "Name", "Age", "Gender", "Blood Type", "Medical Condition",
    "Date of Admission", "Doctor", "Hospital", "Insurance Provider",
    "Billing Amount", "Room Number", "Admission Type", "Discharge Date",
    "Medication", "Test Results"
}

incoming     = set(raw_df.columns)
new_cols     = incoming - EXPECTED_COLS
missing_cols = EXPECTED_COLS - incoming

if new_cols:
    print(f"SCHEMA DRIFT — new columns: {new_cols}  → kept via mergeSchema")
if missing_cols:
    print(f"SCHEMA DRIFT — missing columns: {missing_cols} → filled with null")

for col in missing_cols:
    raw_df = raw_df.withColumn(col, F.lit(None).cast("string"))


## 5 · Rename Columns & Add Metadata


In [0]:
def clean_col(c):
    return re.sub(r'[^a-zA-Z0-9]', '_', c.strip()).lower()

for old in raw_df.columns:
    raw_df = raw_df.withColumnRenamed(old, clean_col(old))

bronze_df = (raw_df
    .withColumn("ingested_at",    F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.lit(SOURCE_PATH))
    .withColumn("source_format",  F.lit(get_ext(SOURCE_PATH)))
    .withColumn("batch_id",       F.lit(BATCH_ID))
)

bronze_df.printSchema()


root
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- medical_condition: string (nullable = true)
 |-- date_of_admission: string (nullable = true)
 |-- doctor: string (nullable = true)
 |-- hospital: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- room_number: string (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- medication: string (nullable = true)
 |-- test_results: string (nullable = true)
 |-- ingested_at: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- source_file: string (nullable = false)
 |-- source_format: string (nullable = false)
 |-- batch_id: string (nullable = false)



## 6 · Write to Bronze Delta Table (Merge Upsert)

**Upsert Logic:**
1. **Deduplicate source** on merge keys (name + date_of_admission + hospital)
   - Source file has ~954 duplicate business keys
   - Must deduplicate BEFORE merge to avoid ambiguity error
2. If table exists → **MERGE** on business keys
   - Matched rows → UPDATE with latest data
   - Unmatched rows → INSERT as new records
3. If table doesn't exist → **CREATE** with initial load
4. mergeSchema = new columns auto-added (schema evolution)

**Why deduplicate on merge keys?**
- Delta Lake merge requires 1:1 mapping (one source row → one target row)
- Multiple source rows matching same target → ambiguity error
- Deduplication removes exact duplicates from raw file


In [0]:
from delta.tables import DeltaTable

# STEP 1: Deduplicate source data on MERGE KEY columns only
# Must match the merge condition: name + date_of_admission + hospital
original_count = bronze_df.count()
bronze_df_dedup = bronze_df.dropDuplicates(["name", "date_of_admission", "hospital"])
dedup_count = bronze_df_dedup.count()
duplicates_removed = original_count - dedup_count

if duplicates_removed > 0:
    print(f"🧼 Removed {duplicates_removed} duplicate rows from source ({original_count} → {dedup_count})")

# STEP 2: Check if table exists
try:
    existing_table = spark.table("de_1.bronze_healthcare")
    table_exists = True
except:
    table_exists = False

# STEP 3: Merge or create
if table_exists:
    # MERGE on business keys: name + date_of_admission + hospital
    bronze_table = DeltaTable.forName(spark, "de_1.bronze_healthcare")
    
    bronze_table.alias("target").merge(
        bronze_df_dedup.alias("source"),
        """target.name = source.name 
           AND target.date_of_admission = source.date_of_admission
           AND target.hospital = source.hospital"""
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print("✅ Bronze upsert complete (merge on business keys)")
else:
    # Table doesn't exist yet - create it with initial load
    bronze_df_dedup.write.format("delta").mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("overwriteSchema", "true") \
        .saveAsTable("de_1.bronze_healthcare")
    
    print("✅ Bronze initial load complete (table created)")

print(f"\nTotal bronze rows: {spark.table('de_1.bronze_healthcare').count()}")

🧼 Removed 954 duplicate rows from source (7589 → 6635)
✅ Bronze upsert complete (merge on business keys)

Total bronze rows: 7589


## 7 · Ingestion Complete


In [0]:
print(f"Ingestion complete: {SOURCE_PATH}")


Ingestion complete: /Volumes/workspace/default/practice/healthcare_2019.csv


## 8 · Schema Evolution Demo
Simulates a future file arriving with a new column.
mergeSchema absorbs it without recreating the table.


In [0]:
demo = (spark.createDataFrame(
    [("demo",)],
    StructType([StructField("new_future_column", StringType(), True)])
).withColumn("ingested_at",    F.current_timestamp())
 .withColumn("ingestion_date", F.current_date())
 .withColumn("source_file",    F.lit("schema_evo_demo"))
 .withColumn("source_format",  F.lit("csv"))
 .withColumn("batch_id",       F.lit("evo_demo")))

demo.write.format("delta").mode("append") \
    .option("mergeSchema", "true").saveAsTable("de_1.bronze_healthcare")

print("Schema evolution: new column merged into bronze")

# Clean up demo row
spark.sql("DELETE FROM de_1.bronze_healthcare WHERE source_file = 'schema_evo_demo'")


Schema evolution: new column merged into bronze


DataFrame[num_affected_rows: bigint]

## 9 · Time Travel — view last 5 versions


In [0]:
spark.sql("DESCRIBE HISTORY de_1.bronze_healthcare") \
     .select("version", "timestamp", "operation") \
     .show(5, truncate=False)


+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|12     |2026-06-22 14:13:30|DELETE   |
|11     |2026-06-22 14:13:25|WRITE    |
|10     |2026-06-22 14:13:19|MERGE    |
|9      |2026-06-17 15:16:28|MERGE    |
|8      |2026-06-17 15:15:49|MERGE    |
+-------+-------------------+---------+
only showing top 5 rows


## 10 · Optimize + Z-Order


In [0]:
spark.sql("OPTIMIZE de_1.bronze_healthcare")
print("Bronze optimized")


Bronze optimized
